# Setting Up

## Importing raw data

In [ ]:
#This gets us into the right directory from home, in order to run the import python script from Mat

#sys = system (module), gives information and control over python interpreter/terminal itself
import sys
sys.path.append("/home/565/pv3484/aus_substation_electricity")

#% is a magic command, special shortcut command that lets you control/interact with notebook environment (ie. gives control of terminal without writing full python code)
%cd /home/565/pv3484/aus_substation_electricity

!pwd

In [ ]:
#This section imports the substations that Mat put together

%run /home/565/pv3484/aus_substation_electricity/import_substation.py

## Lat/lon into info

In [ ]:
from geopy.geocoders import Nominatim
import pandas as pd
import time

# Initialize geocoder
geolocator = Nominatim(user_agent="sydney_demand_mapper")

def get_coords(place):
    """Return (lat, lon) for a suburb name, or (None, None) if not found."""
    try:
        loc = geolocator.geocode(f"{place}, New South Wales, Australia")
        if loc:
            return loc.latitude, loc.longitude
    except Exception as e:
        print(f"Geocoding failed for {place}: {e}")
    return None, None

# Apply geocoding to the 'Name' column
latitudes, longitudes = [], []
for suburb in info['Name']:
    lat, lon = get_coords(suburb)
    latitudes.append(lat)
    longitudes.append(lon)
    time.sleep(1)  # polite pause to avoid hitting API limits

info['latitude'] = latitudes
info['longitude'] = longitudes

In [ ]:
missing = info[info["latitude"].isna() | info["longitude"].isna()]
missing["Name"].unique()
#Dee Why West doesn't exist as a suburb polygon, so will need to change the name to Dee Why

In [ ]:
dee_why_west_lat = -33.73441
dee_why_west_lon = 151.28278
#Found the lat/lon information online

In [ ]:
info.loc[info["Name"] == "Dee Why West", "latitude"] = dee_why_west_lat
info.loc[info["Name"] == "Dee Why West", "longitude"] = dee_why_west_lon
#inputting lat and lon from online into info

## Holiday Function

In [ ]:
import pandas as pd
from datetime import date, timedelta
from dateutil.easter import easter

#Monarch's Birthday
def second_monday_of_june(y):
    """Return the date of the second Monday in June for year y."""
    june = pd.date_range(start=f"{y}-06-01", end=f"{y}-06-30", freq="D")
    mondays = june[june.weekday == 0]   # Monday = 0
    return mondays[1]                   # second Monday



# Define all national public holidays (including moving ones like Easter)
HOLIDAYS_VIC = {
    "New Year's Day": lambda y: pd.Timestamp(f"{y}-01-01"),
    "Australia Day": lambda y: pd.Timestamp(f"{y}-01-26"),
    "Good Friday": lambda y: pd.Timestamp(easter(y)) - pd.Timedelta(days=2),
    "Easter Saturday": lambda y: pd.Timestamp(easter(y)) - pd.Timedelta(days=1),
    "Easter Sunday": lambda y: pd.Timestamp(easter(y)),
    "Easter Monday": lambda y: pd.Timestamp(easter(y)) + pd.Timedelta(days=1),
    "ANZAC Day": lambda y: pd.Timestamp(f"{y}-04-25"),
    "Monarch's Birthday": lambda y: second_monday_of_june(y),
    "Christmas Day": lambda y: pd.Timestamp(f"{y}-12-25"),
    "Boxing Day": lambda y: pd.Timestamp(f"{y}-12-26"),
}

holiday_order = list(HOLIDAYS_VIC.keys())

## Import CSV

In [ ]:
import pandas as pd

rank = pd.read_csv(
    "/home/565/pv3484/aus_substation_electricity/data/cleaned_data/full_nsw_relative_rank.csv"
)


## Time blocks

In [ ]:
blocks = {
    "00_04": range(0, 4),
    "04_10": range(4, 10),
    "10_15": range(10, 15),
    "15_20": range(15, 20),
    "20_24": range(20, 24)
}


## Temperature for plotting

In [ ]:
# Ensure datetime index
obs.index = pd.to_datetime(obs.index)

# Extract hour + date
obs["hour"] = obs.index.hour
obs["date"] = obs.index.date

# Prepare output list
temp_rows = []

# Loop over each day
for day, df_day in obs.groupby("date"):

    row = {"date": pd.Timestamp(day)}

    # Loop over each block
    for block_name, hours in blocks.items():
        block_vals = df_day[df_day["hour"].isin(hours)]["t2m"]

        # Mean temperature for this block
        row[f"{block_name}_temp_mean"] = block_vals.mean()

    temp_rows.append(row)

# Convert to DataFrame
temp_blocks = pd.DataFrame(temp_rows)


In [ ]:
# Ensure datetime
rank["date"] = pd.to_datetime(rank["date"])
temp_blocks["date"] = pd.to_datetime(temp_blocks["date"])

# Merge block-level temperatures into rank
rank = rank.merge(
    temp_blocks,
    on="date",
    how="left"
)


# 1x3 panel map (for each time block)
- public holiday, weekday and weekend
- ranking of the day minus weekend/day
- Difference in ranking for each time block
- mapped with substations above residential fraction of 75%

In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import contextily as ctx

def map_ph_we_wd_block(
    df,
    info,
    holiday,
    block,
    df_station_col="station_code",
    info_station_col="energy_asset",
    lat_col="latitude",        # <-- corrected default
    lon_col="longitude",       # <-- corrected default
    residential_col="Residential",
    residential_threshold=0.75,
    block_suffix="_mean",
    crs_epsg=3857
):

    """
    Creates a 1×3 spatial map for a single time block:
        1. Public Holiday mean rank
        2. PH − Weekend
        3. PH − Weekday

    Fully soft-coded for your actual column names.
    """

    # Filter to high-residential stations
    high_res = info[info[residential_col] >= residential_threshold].copy()
    high_res_codes = high_res[info_station_col].unique()

    df = df[df[df_station_col].isin(high_res_codes)].copy()

    # Filter to the selected holiday
    df_hol = df[df["holiday"] == holiday].copy()

    # Prepare GeoDataFrame
    gdf = high_res.copy()
    gdf = gpd.GeoDataFrame(
        gdf,
        geometry=gpd.points_from_xy(gdf[lon_col], gdf[lat_col]),
        crs="EPSG:4326"
    ).to_crs(crs_epsg)

    # Compute PH, PH-WE, PH-WD for each station
    results = []
    col = f"{block}{block_suffix}"

    for station in gdf[info_station_col]:
        df_s = df_hol[df_hol[df_station_col] == station]
        if df_s.empty:
            continue

        df_ph = df_s[df_s["is_holiday"] == True]
        ph_vals = df_ph.groupby("year")[col].mean().rename("ph")

        df_we = df_s[(df_s["is_holiday"] == False) & (df_s["is_weekend"] == True)]
        we_vals = df_we.groupby("year")[col].mean().rename("we")

        df_wd = df_s[(df_s["is_holiday"] == False) & (df_s["is_weekend"] == False)]
        wd_vals = df_wd.groupby("year")[col].mean().rename("wd")

        merged = (
            ph_vals.to_frame()
            .merge(we_vals, on="year", how="left")
            .merge(wd_vals, on="year", how="left")
        )

        merged["ph_we"] = merged["ph"] - merged["we"]
        merged["ph_wd"] = merged["ph"] - merged["wd"]

        vals = merged.mean()

        results.append({
            info_station_col: station,
            "ph": vals["ph"],
            "ph_we": vals["ph_we"],
            "ph_wd": vals["ph_wd"]
        })

    # Merge results back into GeoDataFrame
    res_df = pd.DataFrame(results)
    gdf = gdf.merge(res_df, on=info_station_col, how="left")

    # Create 1×3 spatial map
    fig, axes = plt.subplots(1, 3, figsize=(18, 6), sharex=True, sharey=True)

    panels = [
        ("Public Holiday Mean Rank", "ph"),
        ("PH − Weekend", "ph_we"),
        ("PH − Weekday", "ph_wd")
    ]

    for ax, (title, colname) in zip(axes, panels):
        gdf.plot(
            ax=ax,
            column=colname,
            cmap="coolwarm",
            legend=True,
            markersize=60,
            edgecolor="black"
        )
        ctx.add_basemap(ax, source=ctx.providers.Stamen.TonerLite)
        ax.set_title(f"{holiday}\n{title} ({block})")
        ax.set_axis_off()

    plt.tight_layout()
    plt.show()


In [ ]:
map_ph_we_wd_block(
    df=rank,
    info=info,
    holiday="Monarch's Birthday",
    block="15_20",
    df_station_col="station_name",
    info_station_col="Name",
    lat_col="latitude",
    lon_col="longitude"
)


In [ ]:
print(rank["station_code"].unique()[:20])
print(info["energy_asset"].unique()[:20])


In [ ]:
print(rank["station_name"].unique()[:20])
print(info["Name"].unique()[:20])
